### Combining PM10

In [1]:
import pandas as pd
import glob
import os


base_folder = "Measurements/PM10"
all_files = glob.glob(os.path.join(base_folder, "*/*.csv")) 

dfs = []

for file in all_files:
    df = pd.read_csv(file)
    
    # Extract year from folder name
    year = os.path.basename(os.path.dirname(file))
    df["year"] = int(year)  # add year column
    
    # Extract state from file name
    state = os.path.splitext(os.path.basename(file))[0]
    df["state"] = state 
    
    dfs.append(df)

# Concatenate all
combined_pm10 = pd.concat(dfs, ignore_index=True)
combined_pm10

,year,state,station,start_time,end_time,value
0,2016,Baden-Württemberg,215,2016-07-28 11:00:00,2016-07-28 12:00:00,16.0
1,2016,Baden-Württemberg,215,2016-07-29 11:00:00,2016-07-29 12:00:00,7.0
2,2016,Baden-Württemberg,215,2016-07-30 11:00:00,2016-07-30 12:00:00,10.0
3,2016,Baden-Württemberg,215,2016-07-31 11:00:00,2016-07-31 12:00:00,10.0
4,2016,Baden-Württemberg,215,2016-08-01 11:00:00,2016-08-01 12:00:00,11.0
...,...,...,...,...,...,...
145721,2025,Thüringen,1934,2025-08-15 11:00:00,2025-08-15 12:00:00,37.0
145722,2025,Thüringen,1934,2025-08-16 11:00:00,2025-08-16 12:00:00,19.0
145723,2025,Thüringen,1934,2025-08-17 11:00:00,2025-08-17 12:00:00,11.0
145724,2025,Thüringen,1934,2025-08-18 11:00:00,2025-08-18 12:00:00,14.0


#### Combining PM25

In [2]:
base_folder = "Measurements/PM25"
all_files = glob.glob(os.path.join(base_folder, "*/*.csv")) 

dfs = []

dfs = []
for file in all_files:
    # Skip empty files
    if os.path.getsize(file) == 0:
        continue

    # Skip files with no header/columns
    with open(file, "r") as f:
        first_line = f.readline().strip()
        if not first_line:  # blank line
            continue

    # Read CSV
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all
combined_pm25 = pd.concat(dfs, ignore_index=True)

combined_pm25

,year,state,station,start_time,end_time,value
0,2016,Bayern,435,2016-07-30 11:00:00,2016-07-30 12:00:00,6.0
1,2016,Bayern,435,2016-07-31 11:00:00,2016-07-31 12:00:00,8.0
2,2016,Bayern,435,2016-08-01 11:00:00,2016-08-01 12:00:00,6.0
3,2016,Bayern,435,2016-08-02 11:00:00,2016-08-02 12:00:00,11.0
4,2016,Bayern,435,2016-08-03 11:00:00,2016-08-03 12:00:00,8.0
...,...,...,...,...,...,...
99139,2025,Thüringen,1934,2025-08-15 11:00:00,2025-08-15 12:00:00,15.0
99140,2025,Thüringen,1934,2025-08-16 11:00:00,2025-08-16 12:00:00,10.0
99141,2025,Thüringen,1934,2025-08-17 11:00:00,2025-08-17 12:00:00,5.0
99142,2025,Thüringen,1934,2025-08-18 11:00:00,2025-08-18 12:00:00,5.0


#### Combining the data frames and cleaning up a bit

In [3]:
# Merge the dataframes
measurements_raw = combined_pm10.merge(
    combined_pm25,
    on=["year","state", "station", "start_time", "end_time"],
    how="outer"
)

# Changing the times to only include the date
measurements_raw = measurements_raw.drop("end_time", axis = 1)
measurements_raw = measurements_raw.rename(columns={"start_time": "date"})
measurements_raw["date"] = pd.to_datetime(measurements_raw["date"]).dt.date

# Rename the value columns respectively
measurements_raw = measurements_raw.rename(columns={"value_x": "pm10"})
measurements_raw = measurements_raw.rename(columns={"value_y": "pm25"})
measurements_raw

,year,state,station,date,pm10,pm25
0,2016,Baden-Württemberg,215,2016-07-28,16.0,NaN
1,2016,Baden-Württemberg,215,2016-07-29,7.0,NaN
2,2016,Baden-Württemberg,215,2016-07-30,10.0,NaN
3,2016,Baden-Württemberg,215,2016-07-31,10.0,NaN
4,2016,Baden-Württemberg,215,2016-08-01,11.0,NaN
...,...,...,...,...,...,...
154961,2025,Thüringen,1934,2025-08-15,37.0,15.0
154962,2025,Thüringen,1934,2025-08-16,19.0,10.0
154963,2025,Thüringen,1934,2025-08-17,11.0,5.0
154964,2025,Thüringen,1934,2025-08-18,14.0,5.0


#### Merging for the means

In [6]:
# Aggregating the stats for each year per state
state_yearly = (
    measurements_raw
    .groupby(["year", "state"])
    .agg({
        "pm10": "mean",
        "pm25": "mean"
    })
    .reset_index()
)

# Aggregating over the whole country
country_yearly = (
    state_yearly
    .groupby("year")
    .agg({
        "pm10": "mean",
        "pm25": "mean"
    })
    .reset_index()
)

state_daily = measurements_raw.drop("station", axis = 1)
country_daily = (
    measurements_raw
    .groupby(["date", "state"])
    .agg({
        "pm10": "mean",
        "pm25": "mean"
    })
    .reset_index()
)

# Saving the data
# Saving the data
state_daily.to_csv("CollectedData/SummerHolidaysState_Daily_Data.csv", index = False)
country_daily.to_csv("CollectedData/SummerHolidaysCountry_Daily_Data.csv", index = False)
state_yearly.to_csv("CollectedData/SummerHolidaysState_Yearly_Data.csv", index = False)
country_yearly.to_csv("CollectedData/SummerHolidaysCountry_Yearly_Data.csv", index = False)